# Iranian REIT returns: four funds and cross assets

Kelid, Danik, Arzesh Maskan, and Kakh are compared with TEDPIX, USD/IRR, and monthly Tehran housing. REIT returns use traded closes and a custom fractional-unit reinvestment rule. Danik has one recorded approved 1,800 IRR distribution; Kelid has approved 907 and 864 IRR distributions. Cash is assumed available on the assembly date and buys units at the first traded close on or afterward. This is a scenario, since actual cash payment timing and complete payout histories are unresolved. Arzesh Maskan and Kakh currently have no approved cash event in this ledger; their scenario equals traded-price performance and should not be called complete total return. No TSETMC adjusted fund price enters these plots or correlation estimates.

The final section examines weekly USD lags over the last 24 Jalali months. Each fund's five lag correlations use identical paired weeks. A lag curve needs at least 74 common weeks (70% window coverage), and a predictive regression needs at least 52 complete weeks. Kakh currently fails both rules.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/processed/analysis/reit_assembly_reinvested_daily.csv').exists())
data = root / 'data/processed/analysis'
fund_order = ['Arzesh Maskan', 'Kelid', 'Danik', 'Kakh']
daily = pd.read_csv(data / 'reit_assembly_reinvested_daily.csv', dtype={'ins_code': str})
events = pd.read_csv(data / 'reit_assembly_reinvestment_events.csv', dtype={'ins_code': str})
weekly = pd.read_csv(data / 'reit_reinvested_weekly_returns.csv')
monthly = pd.read_csv(data / 'reit_reinvested_monthly_returns.csv')
lag_correlations = pd.read_csv(data / 'reit_usd_weekly_lag_correlations.csv')
predictive = pd.read_csv(data / 'reit_usd_weekly_predictive_regressions.csv')
cumulative = pd.read_csv(data / 'reit_usd_tedpix_two_year_cumulative.csv')
housing = pd.read_csv(data / 'tehran_housing_two_year_cumulative.csv')
print(f'Four REITs; {len(events)} recorded distributions; {int(lag_correlations.correlation.notna().sum())} reportable lag points.')

Four REITs; 3 recorded distributions; 15 reportable lag points.


## Two-year cumulative returns

The four REIT lines use custom reinvested traded-close values at observed Friday-ending weeks. Housing remains a separate monthly Kilid listing-price proxy, with no weekly interpolation. Every line starts at zero at its first observation inside the window.

In [2]:
fig = go.Figure()
asset_order = ['TEDPIX', 'USD/IRR'] + fund_order
for name in asset_order:
    sample = cumulative.loc[cumulative.asset == name].sort_values('week_end_gregorian')
    fig.add_trace(go.Scatter(x=sample.week_end_gregorian, y=sample.cumulative_return, name=name, mode='lines', connectgaps=True, customdata=sample[['source_observation_date', 'price_method']].to_numpy(), hovertemplate='%{fullData.name}<br>Week: %{x}<br>Return: %{y:.1%}<br>Observed: %{customdata[0]}<br>Method: %{customdata[1]}<extra></extra>'))
fig.add_trace(go.Scatter(x=housing.observation_date_gregorian, y=housing.cumulative_return, name='Tehran housing (monthly)', mode='lines+markers', line=dict(dash='dash', width=3), hovertemplate='Housing month end: %{x}<br>Return: %{y:.1%}<extra></extra>'))
fig.update_layout(title='Two-year cumulative returns', xaxis_title='Gregorian date', yaxis_title='Cumulative return', yaxis_tickformat='.0%', height=650, hovermode='closest')
fig.show()

## Dividends and reconstructed fund returns

The solid paths compound fractional units purchased with the recorded dividends; dotted paths show the same fund's raw traded-price appreciation. Kelid's two purchases compound. The actual reinvestment trade, price, and resulting unit count are listed beneath the chart.

In [3]:
fig = go.Figure()
for name in fund_order:
    fund = daily.loc[daily.fund == name].sort_values('source_date_gregorian')
    raw = fund.traded_close_irr / fund.traded_close_irr.iloc[0] - 1
    reinvested = fund.reinvested_value_irr / fund.reinvested_value_irr.iloc[0] - 1
    fig.add_trace(go.Scatter(x=fund.source_date_gregorian, y=reinvested, name=f'{name} reinvested', legendgroup=name))
    fig.add_trace(go.Scatter(x=fund.source_date_gregorian, y=raw, name=f'{name} traded', legendgroup=name, line=dict(dash='dot')))
fig.update_layout(title='Custom dividend reinvestment versus traded prices', xaxis_title='Trade date', yaxis_title='Cumulative return since first trade', yaxis_tickformat='.0%', height=650, hovermode='x unified')
fig.show()
display(events[['fund', 'assembly_date_gregorian', 'reinvestment_date_gregorian', 'cash_irr_per_unit', 'reinvestment_close_irr', 'units_after', 'evidence_status']].sort_values(['fund', 'assembly_date_gregorian']))

,fund,assembly_date_gregorian,reinvestment_date_gregorian,cash_irr_per_unit,reinvestment_close_irr,units_after,evidence_status
2,Danik,2025-08-18,2025-08-18,1800,11057.0,1.162793,user_supplied
0,Kelid,2025-10-08,2025-10-11,907,10380.0,1.087380,user_supplied_and_manager_confirmed
1,Kelid,2026-09-16,2026-09-19,864,16461.0,1.144454,user_supplied


## Weekly USD lead-lag analysis

Daily, weekly, and monthly Kelid/Danik returns come from the same compounded-unit values. The tables show their latest observations. The lag plot compares this week's reconstructed REIT return with the USD/IRR return from the same week (`0`) or exactly 1–4 prior completed Friday-ending weeks. Missing weeks remain missing; the five points for each fund use identical dates. USD returns crossing its source-method boundary are excluded. A fund needs at least 74 common paired weeks. This plot describes co-movement, not prediction or causation.

In [4]:
display(weekly.pivot(index='period', columns='fund', values='reinvested_return').reindex(columns=fund_order).tail(8).round(4))
display(monthly.pivot(index='period', columns='fund', values='reinvested_return').reindex(columns=fund_order).tail(8).round(4))

fund,Arzesh Maskan,Kelid,Danik,Kakh
period,,,,
2026-08-14,-0.0320,-0.0292,-0.0351,-0.0018
2026-08-21,-0.0168,-0.0119,-0.0056,0.0068
2026-08-28,0.0991,0.1056,-0.0372,0.0543
2026-09-04,-0.0145,-0.0275,0.0125,-0.0286
2026-09-11,0.1029,0.0448,0.0640,0.0356
2026-09-18,-0.0209,-0.0019,-0.0306,-0.0126
2026-09-25,-0.0168,-0.0369,-0.0213,0.0136
2026-10-02,0.0539,0.0613,0.0634,0.0206


fund,Arzesh Maskan,Kelid,Danik,Kakh
period,,,,
1404/10,-0.0126,-0.0695,-0.1205,NaN
1404/11,0.0769,0.0441,0.0302,NaN
1404/12,-0.1148,-0.0833,-0.0980,NaN
1405/02,NaN,NaN,NaN,NaN
1405/03,0.0103,0.0787,-0.0283,-0.0652
1405/04,-0.0904,0.0256,0.0247,-0.0254
1405/05,0.0570,0.0487,0.0357,0.0802
1405/06,0.1361,0.0652,-0.0371,0.0607


In [5]:
lag_plot = lag_correlations.loc[lag_correlations.correlation.notna()].copy()
fig = go.Figure()
for fund in fund_order:
    sample = lag_plot.loc[lag_plot.fund == fund].sort_values('usd_lag_weeks')
    if sample.empty:
        continue
    count = int(sample.paired_weeks.iloc[0])
    fig.add_trace(go.Scatter(x=sample.usd_lag_weeks, y=sample.correlation, mode='lines+markers', name=f'{fund} (n={count})', customdata=sample[['first_pair_week', 'last_pair_week']].to_numpy(), hovertemplate='%{fullData.name}<br>USD lag: %{x} weeks<br>Pearson r: %{y:+.3f}<br>Paired dates: %{customdata[0]} to %{customdata[1]}<extra></extra>'))
fig.add_hline(y=0, line_color='gray', line_width=1)
fig.update_layout(title='USD/IRR lead-lag correlation on identical paired weeks', xaxis_title='USD lag in completed weeks (0 = same week)', yaxis_title='Pearson correlation', xaxis=dict(tickmode='linear', dtick=1), yaxis=dict(range=[-1, 1]), height=510)
fig.add_annotation(text='Kakh omitted: 16 common weeks, below the 74-week coverage threshold', xref='paper', yref='paper', x=0, y=-0.22, showarrow=False)
fig.show()

## Predictive regression: one- and two-week dollar lags

For each fund, fit the weekly model:

$$r^{REIT}_t=\alpha+\phi r^{REIT}_{t-1}+\beta_1r^{USD}_{t-1}+\beta_2r^{USD}_{t-2}+\gamma r^{TEDPIX}_{t-1}+\varepsilon_t.$$

The table reports the two USD coefficients, their **joint** Wald-test p-value using HAC/Newey-West standard errors with four lags, its Benjamini-Hochberg FDR adjustment across reportable funds, and the in-sample increase in R² versus the same-date model without USD terms. At least 52 complete weekly rows are required. These associations are not causal effects or proof of out-of-sample forecasting skill. Kakh has insufficient history.

In [6]:
rows = predictive.set_index('fund').reindex(fund_order).reset_index().copy()
def number(value, spec):
    return format(value, spec) if pd.notna(value) else '—'
values = [rows.fund, rows.paired_weeks.astype(int),
          rows.beta_usd_l1.map(lambda x: number(x, '+.3f')),
          rows.beta_usd_l2.map(lambda x: number(x, '+.3f')),
          rows.joint_usd_pvalue.map(lambda x: number(x, '.3f')),
          rows.joint_usd_fdr_pvalue.map(lambda x: number(x, '.3f')),
          rows.incremental_r_squared.map(lambda x: number(x, '+.3f')),
          rows.status.map(lambda x: 'Reported' if x == 'reported' else 'Insufficient history')]
fig = go.Figure(go.Table(header=dict(values=['Fund', 'Weeks', 'USD lag 1 β', 'USD lag 2 β', 'Joint p', 'FDR p', 'Δ in-sample R²', 'Status'], fill_color='#183153', font=dict(color='white')), cells=dict(values=values, fill_color='#f3f6fa', height=34)))
fig.update_layout(title='Weekly REIT return model with past USD/IRR returns', height=330)
fig.show()

The lag curve is exploratory; the two-lag regression is the prespecified predictive comparison. The nominal 24-month span is shared, while each regression uses its own complete-case weeks. A low in-sample p-value alone is not a trading rule. The next robustness check is to compare assembly-date with actual payment-date reinvestment once those cash dates and missing distributions are verified.